# Training a DR-PPO Policy

This notebook walks through the DR-PPO training loop with toy data. The real training loop requires Vivado or Quartus rollouts; here we substitute random tensors for fast iteration on the agent's update step.

**Estimated runtime:** ~1 minute on a CPU.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / 'src'))

import torch
torch.manual_seed(42)

from robin_fpga import Agent
from robin_fpga.agent import AgentConfig

## 1. Build the agent

Default config: 2-layer GAT encoder, 192-action policy, value head, slack-regression head.

In [ ]:
agent = Agent(AgentConfig(cvar_beta=0.20, conformal_alpha=0.05))
n_params = sum(p.numel() for p in agent.parameters())
print(f'Agent has {n_params:,} parameters')

## 2. Synthetic batch

In the real loop, this batch comes from `Environment.step()`. Here we fabricate it for a smoke test.

In [ ]:
def make_batch(B=8, N=12):
    return {
        'node_feats': torch.randn(B, N, 16),
        'adj': torch.ones(B, N, N, dtype=torch.bool),
        'tab_feats': torch.randn(B, 32),
        'action': torch.randint(0, 192, (B,)),
        'old_log_prob': torch.randn(B),
        'return': torch.randn(B),
        'wns_target': torch.randn(B) * 0.3,
    }

batch = make_batch()
print({k: v.shape for k, v in batch.items()})

## 3. Run 50 PPO updates

In [ ]:
history = []
for step in range(50):
    batch = make_batch()
    diag = agent.update(batch)
    history.append(diag)
    if step % 10 == 0:
        print(f'[step {step:3d}] total={diag["loss/total"]:.3f} policy={diag["loss/policy"]:.3f} value={diag["loss/value"]:.3f}')

## 4. Plot loss curves

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

h = pd.DataFrame(history)
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
for ax, col in zip(axes.flat, ['loss/total', 'loss/policy', 'loss/value', 'loss/slack']):
    ax.plot(h[col])
    ax.set_title(col)
    ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 5. Save and reload checkpoint

In [ ]:
agent.save('toy_checkpoint.pt')
loaded = Agent.from_checkpoint('toy_checkpoint.pt')
print('Reloaded successfully')